# 🔍 PA2: Multi-Object Tracking Pipeline (Inferência)

**Dupla**: Henrique Coelho Beltrão & Isaias Gouvêa Gonçalves
**Data**: Outubro 2026

Este notebook demonstra o pipeline completo de inferência desenvolvido para o PA2. 
Ele ilustra a integração dos componentes solicitados no edital:
1. Carregamento do Dataset e Detecções (MOT17).
2. Filtro NMS Autoral.
3. Associação de Identidades via Tracker Baseline e Temporal (com RNN).
4. Cálculo das Métricas de Rastreamento (IDF1).
5. Visualização do Rastreamento Final.

*Este formato foi elaborado pensando em uma apresentação executiva de 15 minutos.*

In [1]:
import os
import sys
import torch
from IPython.display import HTML
import matplotlib.pyplot as plt

# Adiciona o diretório raiz ao path para importar a src
sys.path.append(os.path.abspath('..'))

from src.data.mot17 import load_ground_truth, load_detections, load_seqinfo, load_frame_image
from src.tracking.tracker import TemporalTracker, BaselineTracker
from src.tracking.temporal_model import MotionRNN
from src.metrics import evaluate_sequence
from src.visualization import save_tracking_video

## 1. Carregamento dos Dados (MOT17-09-SDP)
Vamos carregar uma sequência do dataset de validação. Carregaremos as detecções, o ground truth e as imagens para podermos rastrear os objetos em cena.

In [2]:
data_root = '../data'
seq_name = 'MOT17-09-SDP'
seq_path = os.path.join(data_root, 'MOT17', 'train', seq_name)

info = load_seqinfo(seq_path)
print(f'Seq Info: {info}')

dets = load_detections(seq_path)
gt_tracks, gt_per_frame = load_ground_truth(seq_path)
print(f'Detecções em {len(dets)} frames.')
print(f'{len(gt_tracks)} trajetórias de Ground Truth.')

Seq Info: {'name': 'MOT17-09-SDP', 'imDir': 'img1', 'frameRate': 30, 'seqLength': 525, 'imWidth': 1920, 'imHeight': 1080, 'imExt': '.jpg'}
Detecções em 525 frames.
26 trajetórias de Ground Truth.


## 2. Pipeline de Tracking: MotionRNN + Associação
O tracker processa o vídeo de maneira online (frame a frame).
Aqui instanciamos o modelo treinado que orienta a associação espacial predizendo o próximo estado da bounding box dos pedestres. O tracker aplica nossa NMS customizada implicitamente (se configurada) e faz o matching ótimo.

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

motion_model = MotionRNN(input_dim=4, hidden_dim=64, cell_type='gru')

ckpt_path = '../checkpoints/motion_gru.pth'
if os.path.exists(ckpt_path):
    motion_model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    print('Checkpoint carregado com sucesso!')
else:
    print('Aviso: Checkpoint não encontrado, rodando com pesos aleatórios para demonstração.')
    
motion_model.to(device)
motion_model.eval()

tracker = TemporalTracker(motion_model=motion_model, iou_threshold=0.3, max_age=30, min_hits=3, matching='hungarian', device=device)

Checkpoint carregado com sucesso!


## 3. Inferência Online (Rastreamento Frame a Frame)
Aqui o algoritmo iterativamente prediz posições e atualiza as identidades. Para brevidade na visualização, processaremos um subconjunto de frames (ex: primeiros 150 frames).

In [4]:
MAX_FRAMES = 150
pred_tracks = {}
vis_frames = []
vis_tracks = []

for frame_id in sorted(dets.keys())[:MAX_FRAMES]:
    frame_dets = dets[frame_id]
    active_tracks = tracker.update(frame_dets)
    
    frame_track_list = []
    for t in active_tracks:
        if t.track_id not in pred_tracks:
            pred_tracks[t.track_id] = {}
        pred_tracks[t.track_id][frame_id] = t.bbox.copy()
        frame_track_list.append({
            'track_id': t.track_id,
            'bbox': t.bbox.copy()
        })
        
    img = load_frame_image(seq_path, frame_id)
    vis_frames.append(img)
    vis_tracks.append(frame_track_list)
    
    if frame_id % 30 == 0:
        print(f'Processado frame {frame_id}/{MAX_FRAMES}...')

print('Inferência finalizada!')

Processado frame 30/150...


Processado frame 60/150...


Processado frame 90/150...


Processado frame 120/150...


Processado frame 150/150...
Inferência finalizada!


## 4. Avaliação Quantitativa (Métricas)
Avaliação local desta sequência usando nossas métricas autorais.

In [5]:
metrics = evaluate_sequence(gt_tracks, pred_tracks, iou_threshold=0.5)

print('=== Resultados da Sequência ===')
print(f'IDF1:             {metrics["idf1"]:.4f}')
print(f'ID Switches:      {metrics["id_switches"]}')
print(f'Fragmentações:    {metrics["fragmentations"]}')
print(f'Erro IDs Únicos:  {metrics["unique_id_error"]}')

=== Resultados da Sequência ===
IDF1:             0.0821
ID Switches:      136
Fragmentações:    101
Erro IDs Únicos:  42


## 5. Visualização: Vídeo do Rastreamento Final
Geramos o vídeo final anotado e o exibimos inline.

In [6]:
output_video_path = '../outputs/tracking_demo.mp4'
os.makedirs('../outputs', exist_ok=True)

save_tracking_video(vis_frames, vis_tracks, output_video_path, fps=15)
print(f'Vídeo salvo em {output_video_path}')

HTML(f"""
<video width="800" controls>
  <source src="{output_video_path}" type="video/mp4">
  Seu navegador não suporta a tag de vídeo.
</video>
""")

Vídeo salvo em ../outputs/tracking_demo.mp4
